<a href="https://colab.research.google.com/github/daria-troshchenko/data-science-projects/blob/main/kudago-events-analysis/CV__kudago_events_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Проведем исследование предстоящих событий в Москве, размещенных на платформе kudago.com


## Наша задача:
* Выгрузить информацию о предстоящих событиях в Москве в датафрейм pandas и сохранить все в csv-файл.

* Провести анализ полученных данных: обработать пустые значения, добавить новые признаки, рассчитать статистические показатели и построить графики.

Для работы в браузере будем использовать библиотеки **selenium** и **requests**.

In [ ]:
from selenium import webdriver as wb
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
import requests
from bs4 import BeautifulSoup
from selenium.common.exceptions import NoSuchElementException, ElementNotInteractableException
import pandas as pd
import re
from time import sleep
import math
import plotly
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
s = Service('D:\chromedriver.exe')
br = wb.Chrome(service = s)
#search_string
br.get('https://kudago.com/msk/')

In [ ]:
submit = br.find_element(By.CSS_SELECTOR, 'a.event-calendar-link')
submit.click()

page1 = br.page_source
soup1 = BeautifulSoup(page1)

Total_Events = int(re.findall(r'\d+', soup1.find_all('span', {'class': 'feed-items-count'})[0].text)[0])
#<span class="feed-items-count">Найдено 1407 событий. </span> на 15.01

urls_event, likes = [], []
urls_event = [link.get('href') for link in soup1.find_all('a', {'class' : 'post-title-link'})]                      #urls_event = [link.get('href') for link in soup1.find_all('a', {'class' : 'post-title-link'}) if 'msk/event/' in link.get('href')]
likes = [l.text.strip() for l in soup1.find_all('span', {'class' : ['post-fav-number js-likes-number one-digit-size', \
                                           'post-fav-number js-likes-number two-digits-size', \
                                           'post-fav-number js-likes-number three-digits-size',\
                                           'post-fav-number js-likes-number four-digits-size',\
                                           'post-fav-number js-likes-number five-digits-size',\
                                           'post-fav-number js-likes-number six-digits-size',\
                                           'post-fav-number js-likes-number seven-digits-size',\
                                           'post-fav-number js-likes-number eight-digits-size']})]

#base - find all links
i = 1
while True:
        try:
            next_p = br.find_element(By.CLASS_NAME, 'load-more-button')
            next_p.click()
            br.implicitly_wait(3)  # задержка в 3 секунды

            page2 = br.page_source
            soup2 = BeautifulSoup(page2)

            urls_event2 = [link.get('href') for link in soup2.find_all('a', {'class' : 'post-title-link'})]
            likes2 = [l2.text.strip() for l2 in soup2.find_all('span', {'class' : ['post-fav-number js-likes-number one-digit-size', \
                                           'post-fav-number js-likes-number two-digits-size', \
                                           'post-fav-number js-likes-number three-digits-size',\
                                           'post-fav-number js-likes-number four-digits-size',\
                                           'post-fav-number js-likes-number five-digits-size',\
                                           'post-fav-number js-likes-number six-digits-size',\
                                           'post-fav-number js-likes-number seven-digits-size',\
                                           'post-fav-number js-likes-number eight-digits-size']})]

            urls_event.extend(urls_event2)
            likes.extend(likes2)
            i += 1
        except ElementNotInteractableException:
            print('End', i)
            break

End 47


In [ ]:
PureUrl, PureLiks = [], [] # del event from https://igoevent.com/
for ind in range(len(urls_event)):
    if ('https://kudago.com/msk/event/' in urls_event[ind]) and urls_event[ind] not in PureUrl:
        PureUrl.append(urls_event[ind])
        PureLiks.append(likes[ind])

Full_url_likes = list(zip(PureUrl, PureLiks))
TotalLoadEvents = len(Full_url_likes)
print(len(PureUrl), len(PureLiks), TotalLoadEvents)
print(len(urls_event), len(likes))

1047 1047 1047
32366 32366


Всего на сайте, в календаре событий, имеется информация о 1415 мероприятии (состояние на 18.01.2022). Однако среди этих событий встречаются такие мероприятия, которые размещены на другой платформе. Например, на сайте igoevent.com. Такие мероприятия мы удаляем из списка поскольку у них другой стиль html-кода, который мы не сможем считать.  

In [ ]:
def GetEwents(url0, likes):
    """
    Returns a tuple with title, abstract, likes, post_views, price, min_age, EventType, type_inline, date, time, place, adress, publication_time, publication_id.
    Parameters:

    url0 is a link to the news (string)
    """
    page0 = requests.get(url0)
    soup0 = BeautifulSoup(page0.text)

    title = soup0.find_all('h1', {'class' : 'post-big-title'})[0].text
    abstract = soup0.find_all('div', {'id' : 'item-description'})[0].text.strip()

    try:
        post_views = soup0.find_all('span', {'class' : 'post-views-number'})[0].text.strip()
    except:
        post_views = float('NAN')

    try:
        price = soup0.find_all('span', {'id' : 'event-price'})[0].text.strip()
    except:
        price = float('NAN')

    try:
        min_age = soup0.find_all('span', {'class': 'age-restriction'})[0].text.strip()
    except:
        min_age = float('NAN')

    try:
        type_inline = soup0.find_all('ul', {'class' : 'post-big-detail-list-inline'})[0].text.strip()
    except:
        type_inline = float('NAN')
    EventType =  [e.text.strip() for e in soup0.find_all('a', {'class' : 'crumbs-list-item-link'})]

    date = [d.text.strip() for d in soup0.find_all('td', {'class' : 'post-schedule-container'})]
    try:
        adress = soup0.find_all('span', {'class' : 'addressItem addressItem--single'})[0].text.strip()
        place = soup0.find_all('a', {'class' : 'post-big-detail-description'})[0].text.strip()
    except:
        if 'Онлайн' in title:
            adress = 'Онлайн'
            place = 'Онлайн'
        else:
            adress = float('NAN')
            place = float('NAN')
    try:
        publication_time = re.findall(r'[^\s][ \w+\d:]+', soup0.find_all('div', {'class' : 'pubdate'})[0].text)[1]
        publication_id = re.findall(r'[^\sA-Y]\d+', soup0.find_all('div', {'class' : 'pubdate'})[0].text)[-1]
    except:
        publication_time = float('NAN')
        publication_id = float('NAN')

    return title, abstract, likes, post_views, price, min_age, EventType, type_inline, date, \
place, adress, publication_time, publication_id

С помощью функции GetEwents соберем всю интересующую нас информацию, пройдя по всем ранее собранным ссылкам на мероприятия:

**title** - название предстоящего мероприятия;

**abstract** - описание мероприятия;

**likes** - количество лайков у мероприятия;

**post_views** - количество просмотров страницы мероприятия;

**min_age** - ограничение по возрасту;

**EventType** - тип мероприятия;

**type_inline** - дополнительная информация о мероприятии (Вид/Жанр/Тематика);

**date** - дата когда состоится мероприятие;

**place** - место проведения;

**adress** - адрес места проведения;

**publication_time** -  дата публикации;

**publication_id** - ID публикации.

In [ ]:
Events = [] # это будет список из кортежей, в которых будут храниться данные по каждом мероприятии
j = 0
for link, like in Full_url_likes:
    res = GetEwents(link, like)
    Events.append(res)
    j += 1
    if j % 100 == 0: print(j)  # проверка, что программа не зависла и данные считываются
    sleep(2) # задержка в 3 секунды

100
200
300
400
500
600
700
800
900
1000


In [ ]:
# 15.01 выгрузила 1186 мероприятий
# 17.01 выгрузила 1021 мероприятий
# 18.01 выгрузила 1047 мероприятий
data = pd.DataFrame(Events)
data['Urls'] = [link for link, like in Full_url_likes]
data.columns = ['Событие', 'Описание', 'Количество лайков', 'Количество просмотров', 'Цена', 'Ограничения по возрасту', 'Категория события', 'Дополнительная информация: вид, жанр, тематика мероприятия', 'Дата события', 'Место проведения', 'Адрес', 'Дата публикации', 'ID публикации', 'Ссылка']

#Сохранение датафрейма:
data.to_csv('events_kudago1801.csv')
size = data.shape
data.head(10)

,Событие,Описание,Количество лайков,Количество просмотров,Цена,Ограничения по возрасту,Категория события,"Дополнительная информация: вид, жанр, тематика мероприятия",Дата события,Место проведения,Адрес,Дата публикации,ID публикации,Ссылка
0,Спектакль «Ракеты взлетают и разбиваются рассы...,"Спектакль-конструктор, который никогда не повт...",15,6177,от 1100,18+,"[Москва, Спектакли]",Драматические постановки\nЭкспериментальный те...,[24 января],Театрально-культурный центр имени Вс. Мейерхольда,"ул. Новослободская, д. 23",17 сентября 2020 19:37,189109,https://kudago.com/msk/event/teatr-raketyi-vzl...
1,Органный концерт «Зимний вечер с друзьями: сак...,Слушатели насладятся новой музыкальной програм...,1,698,от 1000 до 2700,6+,"[Москва, Концерты, Новый год]",Классическая,[21 января],Римско-католический Кафедральный собор Непороч...,"ул. Малая Грузинская, д. 27/13",30 ноября 2021 11:55,194963,https://kudago.com/msk/event/kontsert-zimnij-v...
2,Cтендап-шоу от Stand Up Afisha с бесплатным вх...,Свежие шутки от опытных комиков и восходящих з...,144,120882,"вход бесплатный, депозит на еду — 500",18+,"[Москва, Развлечения, Новый год]",Стэндап,"[18–20 января, 21–22 января, 23 января]",NaN,NaN,28 января 2021 13:01,190707,https://kudago.com/msk/event/entertainment-ste...
3,Концерт «Шедевры классики в соборе» в Римско-к...,"В исполнении струнного квартета, органа и мужс...",2,417,от 900 до 2500,6+,"[Москва, Концерты]",Классическая,[22 января],Римско-католический Кафедральный собор Непороч...,"ул. Малая Грузинская, д. 27/13",30 декабря 2021 11:43,195540,https://kudago.com/msk/event/kontsert-shedevry...
4,Фестиваль «Снег и лёд в Москве»,Зима — время чудес и волшебства! В этом легко ...,10,1943,Бесплатно!,0+,"[Москва, Развлечения, Фестивали, Детям, Новый ...",Open Air\nГородские,[До 28 февраля],Парк Горького (ЦПКиО имени Горького),"ул. Крымский Вал, д. 9",14 января 2022 13:09,195692,https://kudago.com/msk/event/festival-sneg-i-l...
5,Идеальное свидание в лофт-пространстве Рarty Hard,"Близится 14 февраля, а значит, и повод порадов...",,0,от 7700,18+,"[Москва, Развлечения, Праздники, День святого ...",NaN,[До 15 февраля],NaN,NaN,18 января 2022 10:25,195765,https://kudago.com/msk/event/prazdnik-idealnoe...
6,Спектакль «Горбачёв» в Театре Наций,Спектакль о Михаиле и Раисе Горбачёвых от режи...,14,7572,от 1500,16+,"[Москва, Спектакли]",Драматические постановки\nСпектакли с известны...,[14 февраля],Театр Наций,"Петровский пер., д. 3",5 ноября 2020 17:05,189802,https://kudago.com/msk/event/teatr-gorbachyov/
7,Всероссийская образовательная выставка «Навига...,На выставке «Навигатор» старшеклассники и выпу...,,16,Бесплатно!,12+,"[Москва, Прокачай себя, Выставки, Обучение]",Прочее,[5–6 февраля],Центр международной торговли,"Краснопресненская набережная, дом 12, подъезд 4",18 января 2022 11:52,195727,https://kudago.com/msk/event/detyam-navigator/
8,Выставка «Сальвадор Дали & Пабло Пикассо»,Сад имени Баумана при поддержке департамента к...,1685,488574,от 150 до 800,6+,"[Москва, Экскурсии, Выставки, Новый год]","Картины, живопись, графика",[До 22 мая],Путевой дворец Василия Третьего,"ул. Старая Басманная, д. 15, стр. 3",5 октября 2018 19:00,173602,https://kudago.com/msk/event/vyistavka-salvado...
9,Интерактивная выставка «Фанерон. Город мечты»,"Новогодний проект, посвящённый городу будущего...",16,8848,от 1000,6+,"[Москва, Развлечения, Выставки, Детям, Новый год]",Детские\nИнтерактивные,[3–18 января],ЦВЗ «Манеж»,"пл. Манежная, д. 1",24 декабря 2021 16:02,195503,https://kudago.com/msk/event/vyistavka-faneron...


Подгрузим сохраненные ранее данные, поскольку сайт, после многочисленны обращений, стал плохо подгружать данные (не открываются страницы). Приведем полученные данные к типам с которыми можно будет работать.

In [ ]:
# запускать только в случае загрузки данных
data = pd.read_csv('events_kudago1801.csv')
size = data.shape
data.columns = ['№', 'Событие', 'Описание', 'Количество лайков', 'Количество просмотров', 'Цена', 'Ограничения по возрасту', 'Категория события', 'Дополнительная информация: вид, жанр, тематика мероприятия', 'Дата события', 'Место проведения', 'Адрес', 'Дата публикации', 'ID публикации', 'Ссылка']
data['№'] = [n for n in range(1, size[0]+1)]
data['Количество лайков'] = data['Количество лайков'].apply(lambda x: 0 if math.isnan(x) else int(x))
data['Количество просмотров'] = data['Количество просмотров'].apply(lambda x: 0 if math.isnan(x) else int(x))
data['Количество просмотров'] = data['Количество просмотров'].apply(lambda x: 0 if math.isnan(x) else int(x))
data.head(5)

,№,Событие,Описание,Количество лайков,Количество просмотров,Цена,Ограничения по возрасту,Категория события,"Дополнительная информация: вид, жанр, тематика мероприятия",Дата события,Место проведения,Адрес,Дата публикации,ID публикации,Ссылка
0,1,Спектакль «Ракеты взлетают и разбиваются рассы...,"Спектакль-конструктор, который никогда не повт...",15,6177,от 1100,18+,"['Москва', 'Спектакли']",Драматические постановки\nЭкспериментальный те...,['24 января'],Театрально-культурный центр имени Вс. Мейерхольда,"ул. Новослободская, д. 23",17 сентября 2020 19:37,189109.0,https://kudago.com/msk/event/teatr-raketyi-vzl...
1,2,Органный концерт «Зимний вечер с друзьями: сак...,Слушатели насладятся новой музыкальной програм...,1,698,от 1000 до 2700,6+,"['Москва', 'Концерты', 'Новый год']",Классическая,['21 января'],Римско-католический Кафедральный собор Непороч...,"ул. Малая Грузинская, д. 27/13",30 ноября 2021 11:55,194963.0,https://kudago.com/msk/event/kontsert-zimnij-v...
2,3,Cтендап-шоу от Stand Up Afisha с бесплатным вх...,Свежие шутки от опытных комиков и восходящих з...,144,120882,"вход бесплатный, депозит на еду — 500",18+,"['Москва', 'Развлечения', 'Новый год']",Стэндап,"['18–20 января', '21–22 января', '23 января']",NaN,NaN,28 января 2021 13:01,190707.0,https://kudago.com/msk/event/entertainment-ste...
3,4,Концерт «Шедевры классики в соборе» в Римско-к...,"В исполнении струнного квартета, органа и мужс...",2,417,от 900 до 2500,6+,"['Москва', 'Концерты']",Классическая,['22 января'],Римско-католический Кафедральный собор Непороч...,"ул. Малая Грузинская, д. 27/13",30 декабря 2021 11:43,195540.0,https://kudago.com/msk/event/kontsert-shedevry...
4,5,Фестиваль «Снег и лёд в Москве»,Зима — время чудес и волшебства! В этом легко ...,10,1943,Бесплатно!,0+,"['Москва', 'Развлечения', 'Фестивали', 'Детям'...",Open Air\nГородские,['До 28 февраля'],Парк Горького (ЦПКиО имени Горького),"ул. Крымский Вал, д. 9",14 января 2022 13:09,195692.0,https://kudago.com/msk/event/festival-sneg-i-l...


In [ ]:
size = data.shape
size

(1047, 15)

In [ ]:
data['№'] = [n for n in range(1, size[0]+1)]
data['Количество лайков'] = data['Количество лайков'].apply(lambda x: int(x) if str(x).isdigit() else 0)
data['Количество просмотров'] = data['Количество просмотров'].apply(lambda x: int(x) if str(x).isdigit() else 0)
data['Количество просмотров'] = data['Количество просмотров'].apply(lambda x: int(x) if str(x).isdigit() else 0)
data.head(5)

,№,Событие,Описание,Количество лайков,Количество просмотров,Цена,Ограничения по возрасту,Категория события,"Дополнительная информация: вид, жанр, тематика мероприятия",Дата события,Место проведения,Адрес,Дата публикации,ID публикации,Ссылка
0,1,Спектакль «Ракеты взлетают и разбиваются рассы...,"Спектакль-конструктор, который никогда не повт...",15,6177,от 1100,18+,"['Москва', 'Спектакли']",Драматические постановки\nЭкспериментальный те...,['24 января'],Театрально-культурный центр имени Вс. Мейерхольда,"ул. Новослободская, д. 23",17 сентября 2020 19:37,189109.0,https://kudago.com/msk/event/teatr-raketyi-vzl...
1,2,Органный концерт «Зимний вечер с друзьями: сак...,Слушатели насладятся новой музыкальной програм...,1,698,от 1000 до 2700,6+,"['Москва', 'Концерты', 'Новый год']",Классическая,['21 января'],Римско-католический Кафедральный собор Непороч...,"ул. Малая Грузинская, д. 27/13",30 ноября 2021 11:55,194963.0,https://kudago.com/msk/event/kontsert-zimnij-v...
2,3,Cтендап-шоу от Stand Up Afisha с бесплатным вх...,Свежие шутки от опытных комиков и восходящих з...,144,120882,"вход бесплатный, депозит на еду — 500",18+,"['Москва', 'Развлечения', 'Новый год']",Стэндап,"['18–20 января', '21–22 января', '23 января']",NaN,NaN,28 января 2021 13:01,190707.0,https://kudago.com/msk/event/entertainment-ste...
3,4,Концерт «Шедевры классики в соборе» в Римско-к...,"В исполнении струнного квартета, органа и мужс...",2,417,от 900 до 2500,6+,"['Москва', 'Концерты']",Классическая,['22 января'],Римско-католический Кафедральный собор Непороч...,"ул. Малая Грузинская, д. 27/13",30 декабря 2021 11:43,195540.0,https://kudago.com/msk/event/kontsert-shedevry...
4,5,Фестиваль «Снег и лёд в Москве»,Зима — время чудес и волшебства! В этом легко ...,10,1943,Бесплатно!,0+,"['Москва', 'Развлечения', 'Фестивали', 'Детям'...",Open Air\nГородские,['До 28 февраля'],Парк Горького (ЦПКиО имени Горького),"ул. Крымский Вал, д. 9",14 января 2022 13:09,195692.0,https://kudago.com/msk/event/festival-sneg-i-l...


Добавим несколько новых признаков, значения которых будут расчитываться следующим образом.

Для признаков **Минимальная цена** и **Максимальная цена**:
* Напишем функции, которые будут забирать минимальную и макцимальную цену, а потом применим ее к колонке "Цена".
* Заменим в полученых признаках NaN соответсыующими медианными значениями.

Признак **Средняя цена**:
* Используя скоректированные признаки 'Минимальная цена' и 'Максимальная цена' найдем среднюю цену кадого мероприятия.

**Возрастная категория**:
* Напишем функцию, которая сгрупирует 'Ограничения по возрасту' по категориям.

**Формат мероприятия**:
* Применим lambda-функцию, которая определит форммат проведения меропириятий.

**Период события**:
* Применим lambda-функцию, которая определит периодичность проведения меропириятий.

**Год публикации**:
* Напишем функцию, которая найдет год размещения публикации на сайте. Заменим значения NaN модой.

In [ ]:
def return_min_price(full_price):
    """
    Returns a min price.
    Parameters:

    full_price is a informatin about event price (string)
    """
    try:
        p = full_price.split(' ')
        if (len(p) == 1) and p[0] == ('Бесплатно!'):
            return 0
        elif (len(p) == 1) and (p != 'Бесплатно!'):
            return int(p[0])
        if (len(p) == 2) and p[0] == ('от'):
            return int(p[1])
        if (len(p) == 4) and p[0] == ('от'):
            return int(p[1])
        if (len(p) > 4) and 'от' in p:
            return int(p[p.index('от') + 1])
        else:
            return float('NAN')
    except:
        return float('NAN')

In [ ]:
def return_max_price(full_price):
    """
    Returns a max price.
    Parameters:

    full_price is a informatin about event price (string)
    """
    try:
        p = full_price.split(' ')
        if (len(p) == 1) and p[0] == ('Бесплатно!'):
            return 0
        elif (len(p) == 1) and (p != 'Бесплатно!'):
            return int(p[0])
        if (len(p) == 2) and p[0] == ('до'):
            return int(p[1])
        if (len(p) == 4) and p[2] == ('до'):
            return int(p[3])
        elif (len(p) == 4) and p[2] == (''):
            return int(p[3])
        else:
            return float('NAN')
    except:
        return float('NAN')

In [ ]:
def return_age_category(age):
    """
    Returns the age category to which the event belongs.
    Parameters:

    age is a informatin about the age limit of event (string)
    """
    if age in ['0+', '6+']:
        return 'Для детей'
    elif age in ['12+', '16+']:
        return 'Для всей семьи'
    elif age in ['18+']:
        return 'Для взрослых'
    else:
        return float('NAN')


In [ ]:
def return_year_pub(x):
    """
    Returns the year the event was posted on the website.
    Parameters:

    x is a informatin about the age limit of event (string)
    """
    try:
        return re.findall(r'\d{4}', x)[0]
    except:
        return float('NAN')

In [ ]:
data['Минимальная цена'] = data['Цена'].apply(return_min_price)
data['Максимальная цена'] = data['Цена'].apply(return_max_price)

data['Минимальная цена'] = data['Минимальная цена'].fillna(data['Минимальная цена'].median())
data['Максимальная цена'] = data['Максимальная цена'].fillna(data['Максимальная цена'].median())

data['Средняя цена'] = (data['Минимальная цена'] + data['Максимальная цена']) / 2

data['Возрастная категория'] = data['Ограничения по возрасту'].apply(return_age_category)

data['Формат мероприятия'] = data['Место проведения'].apply(lambda x: 'Офлайн' if x !='Онлайн' else x)

data['Период события'] = data['Дата события'].apply(lambda x: 'Постоянно' if 'Круглый год' in x else 'Временно')

data['Год публикации'] = data['Дата публикации'].apply(return_year_pub)
data['Год публикации'] = data['Год публикации'].fillna(data['Год публикации'].mode()[0])
data.head(10)

,№,Событие,Описание,Количество лайков,Количество просмотров,Цена,Ограничения по возрасту,Категория события,"Дополнительная информация: вид, жанр, тематика мероприятия",Дата события,...,Дата публикации,ID публикации,Ссылка,Минимальная цена,Максимальная цена,Средняя цена,Возрастная категория,Формат мероприятия,Период события,Год публикации
0,1,Спектакль «Ракеты взлетают и разбиваются рассы...,"Спектакль-конструктор, который никогда не повт...",15,6177,от 1100,18+,"['Москва', 'Спектакли']",Драматические постановки\nЭкспериментальный те...,['24 января'],...,17 сентября 2020 19:37,189109.0,https://kudago.com/msk/event/teatr-raketyi-vzl...,1100.0,2000.0,1550.0,Для взрослых,Офлайн,Временно,2020
1,2,Органный концерт «Зимний вечер с друзьями: сак...,Слушатели насладятся новой музыкальной програм...,1,698,от 1000 до 2700,6+,"['Москва', 'Концерты', 'Новый год']",Классическая,['21 января'],...,30 ноября 2021 11:55,194963.0,https://kudago.com/msk/event/kontsert-zimnij-v...,1000.0,2700.0,1850.0,Для детей,Офлайн,Временно,2021
2,3,Cтендап-шоу от Stand Up Afisha с бесплатным вх...,Свежие шутки от опытных комиков и восходящих з...,144,120882,"вход бесплатный, депозит на еду — 500",18+,"['Москва', 'Развлечения', 'Новый год']",Стэндап,"['18–20 января', '21–22 января', '23 января']",...,28 января 2021 13:01,190707.0,https://kudago.com/msk/event/entertainment-ste...,800.0,2000.0,1400.0,Для взрослых,Офлайн,Временно,2021
3,4,Концерт «Шедевры классики в соборе» в Римско-к...,"В исполнении струнного квартета, органа и мужс...",2,417,от 900 до 2500,6+,"['Москва', 'Концерты']",Классическая,['22 января'],...,30 декабря 2021 11:43,195540.0,https://kudago.com/msk/event/kontsert-shedevry...,900.0,2500.0,1700.0,Для детей,Офлайн,Временно,2021
4,5,Фестиваль «Снег и лёд в Москве»,Зима — время чудес и волшебства! В этом легко ...,10,1943,Бесплатно!,0+,"['Москва', 'Развлечения', 'Фестивали', 'Детям'...",Open Air\nГородские,['До 28 февраля'],...,14 января 2022 13:09,195692.0,https://kudago.com/msk/event/festival-sneg-i-l...,0.0,0.0,0.0,Для детей,Офлайн,Временно,2022
5,6,Идеальное свидание в лофт-пространстве Рarty Hard,"Близится 14 февраля, а значит, и повод порадов...",0,0,от 7700,18+,"['Москва', 'Развлечения', 'Праздники', 'День с...",NaN,['До 15 февраля'],...,18 января 2022 10:25,195765.0,https://kudago.com/msk/event/prazdnik-idealnoe...,7700.0,2000.0,4850.0,Для взрослых,Офлайн,Временно,2022
6,7,Спектакль «Горбачёв» в Театре Наций,Спектакль о Михаиле и Раисе Горбачёвых от режи...,14,7572,от 1500,16+,"['Москва', 'Спектакли']",Драматические постановки\nСпектакли с известны...,['14 февраля'],...,5 ноября 2020 17:05,189802.0,https://kudago.com/msk/event/teatr-gorbachyov/,1500.0,2000.0,1750.0,Для всей семьи,Офлайн,Временно,2020
7,8,Всероссийская образовательная выставка «Навига...,На выставке «Навигатор» старшеклассники и выпу...,0,16,Бесплатно!,12+,"['Москва', 'Прокачай себя', 'Выставки', 'Обуче...",Прочее,['5–6 февраля'],...,18 января 2022 11:52,195727.0,https://kudago.com/msk/event/detyam-navigator/,0.0,0.0,0.0,Для всей семьи,Офлайн,Временно,2022
8,9,Выставка «Сальвадор Дали & Пабло Пикассо»,Сад имени Баумана при поддержке департамента к...,1685,488574,от 150 до 800,6+,"['Москва', 'Экскурсии', 'Выставки', 'Новый год']","Картины, живопись, графика",['До 22 мая'],...,5 октября 2018 19:00,173602.0,https://kudago.com/msk/event/vyistavka-salvado...,150.0,800.0,475.0,Для детей,Офлайн,Временно,2018
9,10,Интерактивная выставка «Фанерон. Город мечты»,"Новогодний проект, посвящённый городу будущего...",16,8848,от 1000,6+,"['Москва', 'Развлечения', 'Выставки', 'Детям',...",Детские\nИнтерактивные,['3–18 января'],...,24 декабря 2021 16:02,195503.0,https://kudago.com/msk/event/vyistavka-faneron...,1000.0,2000.0,1500.0,Для детей,Офлайн,Временно,2021


## Проведем анализ полученных данных

Применим к новым и наиболее важным признакам метод describe(), который покажет нам основные статистический оценки.

In [ ]:
data[['Событие', 'Количество лайков', 'Количество просмотров', 'Минимальная цена', 'Максимальная цена',\
      'Средняя цена', 'Возрастная категория', 'Формат мероприятия', 'Период события','Год публикации']].describe()

,Количество лайков,Количество просмотров,Минимальная цена,Максимальная цена,Средняя цена
count,1047.000000,1047.000000,1047.000000,1047.000000,1047.000000
mean,37.071633,9174.256925,1537.397326,3023.891117,2280.644222
std,133.309985,39544.591424,4456.335271,5963.237584,4511.811643
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,90.000000,500.000000,2000.000000,1245.000000
50%,4.000000,939.000000,800.000000,2000.000000,1400.000000
75%,20.000000,5464.500000,1400.000000,2000.000000,2000.000000
max,2452.000000,607257.000000,81000.000000,81000.000000,70000.000000


Очевидно, что среди всех запланированных в Москве событий на 2022 год только малая часть пользуется успехом. Среднее значение лайков на одно событие - 37, в то время как среднее количество людей, которые просматривают информацию (т.е. тех, кто заинтересовался) в разы больше - 9174. Средние значения минимальной и максимальной стоимости событий в приделах нормы.

Посмотрим количество событий, организованных для отдельных категорий граждан.

In [ ]:
data['Возрастная категория'].value_counts()

,count
Возрастная категория,
Для всей семьи,534
Для детей,314
Для взрослых,193


Очевидно, что заболеваемость в городе не прогрессирует (состояние на дату выгрузки данных), поскольку практически все события запланированы в формате офлайн. По сравнению с общим количеством событий, запланированных в календаре, неучтенная разница, в которой могут присутствовать онлайн-мероприятия, не существенна (всего 1415 мероприятии на 18.01.2022). Интересно, что в выгрузке предыдущих дней (17.01) присутствовало порядка 170 онлайн мероприятий. Делать выводы рано, но положительная тенденция наблюдается:)

In [ ]:
data['Формат мероприятия'].value_counts()

,count
Формат мероприятия,
Офлайн,1047


Преобладающее количество событий имеют сезонный характер, что видно по следующим данным

In [ ]:
data['Период события'].value_counts()

,count
Период события,
Временно,959
Постоянно,88


Отсортируем наши данные по году публикации на сайте каждого мероприятия и перейдем к построению  графиков.

In [ ]:
data = data.sort_values(by = ['Год публикации'])

In [ ]:
import numpy as np
trace_Events = go.Scatter(x = data['Средняя цена'].tolist(),
                          y = data['Количество лайков'].tolist(),
                          text = data['Событие'],
                          mode = 'markers',
                          marker = {'size' : 11, 'color': 'green'})
our_data =[trace_Events]

our_layout = go.Layout(title = 'Cоотношение средей цены к качеству мероприятий',
                      xaxis = {'title': 'Средняя цена',
                              'range' : [0,7500]},
                      yaxis = {'title': 'Количество лайков'})

fig = go.Figure(data = our_data, layout = our_layout)
fig.add_scatter(x = [data.sort_values(by = ['Средняя цена'])['Средняя цена'].median()]*2, y=np.linspace(-100,2500,2), name='Медианное значение средней цены')

fig.show()

In [ ]:
import plotly.express as px
df = px.data.iris()
fig = px.scatter(df, x=data['Средняя цена'], y=data['Количество лайков'], marginal_x="histogram", marginal_y="rug")
fig.update_layout(title = 'Cоотношение средей цены к качеству мероприятий',
                      xaxis = {'title': 'Средняя цена'},
                      yaxis = {'title': 'Количество лайков'})
fig.show()

На рисунке представлено распределение средней цены мероприятий к количеству проставленных лайков от пользователей. При этом медианное значение для представленного набора данных принимает значение 1400 р. Кроме того, очевидно, что наиболее качественные события находятся в приделах ценового сегмента: от 0 до 4000 р.

Посмотрим какое количество публикаций опубликовано на сайте по годам и как это количество распределено между категориями участников.

In [ ]:
data.groupby('Возрастная категория')['Год публикации'].value_counts()

Возрастная категория  Год публикации
Для взрослых          2021               58
                      2017               22
                      2018               22
                      2020               21
                      2019               19
                      2013               14
                      2022               14
                      2016               11
                      2014                6
                      2015                6
Для всей семьи        2021              135
                      2013               78
                      2020               66
                      2019               63
                      2022               50
                      2016               33
                      2015               30
                      2017               28
                      2018               27
                      2014               24
Для детей             2021              115
                      2013               53
                      2019               28
                      2020               27
                      2015               21
                      2018               19
                      2022               16
                      2017               14
                      2014               13
                      2016                8
Name: count, dtype: int64

Отобразим полученные данные в виде столбчатой диаграммы

In [ ]:
data['Год публикации'].value_counts().reset_index(name = 'Количество мероприятий').sort_values(by = ['Количество мероприятий'])

,Год публикации,Количество мероприятий
9,2014,46
8,2016,52
7,2015,58
6,2017,64
5,2018,69
4,2022,80
3,2019,110
2,2020,114
1,2013,145
0,2021,309


In [ ]:
all_f = data['Год публикации'].value_counts().reset_index(name = 'Количество мероприятий').sort_values(by = ['Количество мероприятий'])
x_year = all_f['Год публикации'].tolist()
y_count = all_f['Количество мероприятий'].tolist()

trace_all = go.Bar(x = x_year, y = y_count,
                   marker = {'color': 'light blue'},
                   name = 'Все мероприятия')

y_countFF = data[(data['Возрастная категория'] == 'Для всей семьи')] ['Год публикации'].value_counts().reset_index(name = 'Количество мероприятий').sort_values(by = ['Количество мероприятий'])['Количество мероприятий'].tolist()
trace_FF = go.Bar(x = x_year, y = y_countFF,
                   marker = {'color': 'green'},
                   name = 'Мероприятия для всей семьи')

y_countCh = data[(data['Возрастная категория'] == 'Для детей')] ['Год публикации'].value_counts().reset_index(name = 'Количество мероприятий').sort_values(by = ['Количество мероприятий'])['Количество мероприятий'].tolist()
trace_Children = go.Bar(x = x_year, y = y_countCh,
                   marker = {'color': 'orange'},
                   name = 'Мероприятия для детей')

y_countAd = data[(data['Возрастная категория'] == 'Для взрослых')] ['Год публикации'].value_counts().reset_index(name = 'Количество мероприятий').sort_values(by = ['Количество мероприятий'])['Количество мероприятий'].tolist()
trace_Adult = go.Bar(x = x_year, y = y_countAd,
                   marker = {'color': 'red'},
                   name = 'Мероприятия для совершеннолетних')

our_data = [trace_all, trace_FF, trace_Children, trace_Adult]
our_layout = go.Layout(title = 'Количество размещенных мероприятий на сайте kudago.com по годам',
                      xaxis = {'title': 'Год'},
                      yaxis = {'title': 'Количество мероприятий'})

fig = go.Figure(data = our_data, layout = our_layout)

fig.show()


Покажем процентное соотношение размещенных мероприятий на сайте kudago.com по годам. Видно, что около 50 % мероприятий опубликованы в период с 2019 по 2021 г.

In [ ]:
fig = go.Figure()
fig.add_trace(go.Pie(values=y_count, labels=x_year, sort = False))
fig.show()

In [ ]:
#trace0 = go.Scatter(y =data['Ограничения по возрасту'].tolist(), x = data['Год публикации'].tolist(),
#trace0 = go.Scatter(y =data['Средняя цена'].tolist(), x = data['Год публикации'].tolist(),
trace0 = go.Scatter(y =data['Количество лайков'].tolist(), x = data['Количество просмотров'].tolist(),
                   mode = 'markers',
                   marker = dict(size = data['Количество лайков'] /10,
                                color =  data['Количество просмотров'],
                                opacity = 0.4,
                                colorscale = 'Inferno',#''Electric',
                                showscale = True),
                   text = data['Событие'],
                   hovertemplate =
                    '<b> %{text}</b>' +
                    '<br><i> Количество просмотров</i>: %{x}' +
                    '<br><i> Количество лайков </i>: %{y}')
                    #'<br><i> Количество просмотров </i>: %{marker.color}' +
                    #'<br><i> Количество лайков</i>: %{marker.size}'
our_data = [trace0]

out_layout = go.Layout(title = 'Популярные события в Москве на платформе kudago.com',
                  xaxis = dict(title = "Количество просмотров"),
                  yaxis = dict(title = "Количество лайков"))

fig = go.Figure( data = our_data, layout = out_layout)
fig.show()

На рисунке представлены самые популярные мероприятия в Москве. Берем на заметку:)